In [ ]:
import os
import time
import json
import threading
import requests
import pandas as pd
from shapely.geometry import shape, Point
from google.colab import drive
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, year, upper, substring, sum as spark_sum

# Mount Drive
drive.mount('/content/drive')

# Constants
BASE_PATH = "/content/drive/MyDrive/Big_Data_Group/Group_Project"
STREAM_PATH = f"{BASE_PATH}/csv/"
OUTPUT_PATH = f"{BASE_PATH}/output/"
CHECKPOINT_PATH = f"{BASE_PATH}/checkpoint_stream/"
SRC_DATA_PATH = f"{BASE_PATH}/src_data"

os.makedirs(STREAM_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(CHECKPOINT_PATH, exist_ok=True)

# Initialize Spark
spark = SparkSession.builder \
    .appName("TorontoBusinessAnalytics") \
    .getOrCreate()

print("✅ Environment and Spark Session Ready")

Mounted at /content/drive
✅ Environment and Spark Session Ready


In [ ]:
def fetch_data_loop():
    """Background thread to pull data from Toronto Open Data Portal every 24h."""
    base_url = "https://ckan0.cf.opendata.inter.prod-toronto.ca"
    package_id = "municipal-licensing-and-standards-business-licences-and-permits"

    while True:
        try:
            url = f"{base_url}/api/3/action/package_show"
            package = requests.get(url, params={"id": package_id}).json()

            for resource in package["result"]["resources"]:
                if resource["datastore_active"]:
                    data_url = f"{base_url}/datastore/dump/{resource['id']}"
                    csv_data = requests.get(data_url).text
                    file_name = f"{STREAM_PATH}data_{int(time.time())}.csv"
                    with open(file_name, "w") as f:
                        f.write(csv_data)
            print(f"✅ {time.strftime('%H:%M:%S')} - New data saved to {STREAM_PATH}")
        except Exception as e:
            print(f"❌ Ingestion Error: {e}")
        time.sleep(86400) # Check again after 24 hours

# Start Background Thread
ingestion_thread = threading.Thread(target=fetch_data_loop, daemon=True)
ingestion_thread.start()
print("🚀 Background ingestion started")

🚀 Background ingestion started


In [ ]:
# Install PySpark (only needed once per runtime)
# !pip install pyspark

# Create Spark session
spark = SparkSession.builder \
    .appName("TorontoGroupProject") \
    .getOrCreate()

print("✅ Spark is ready")

✅ Spark is ready


In [ ]:
folder_path = "/content/drive/MyDrive/Big_Data_Group/Group_Project/csv/"

# Get all CSV files that match the pattern
csv_files = [
    f for f in os.listdir(folder_path)
    if f.startswith("data_") and f.endswith(".csv")
]

# Sort by timestamp (filename)
csv_files.sort(reverse=True)

# Get latest file
latest_file = csv_files[0]

latest_path = os.path.join(folder_path, latest_file)

print("Latest file:", latest_path)

Latest file: /content/drive/MyDrive/Big_Data_Group/Group_Project/csv/data_trigger_1775681065.csv


In [ ]:
def get_streaming_schema():
    """Reads the latest available CSV to infer a stable schema for the stream."""
    csv_files = [f for f in os.listdir(STREAM_PATH) if f.endswith(".csv")]
    if not csv_files:
        raise Exception("No data files found to infer schema!")

    latest_file = sorted(csv_files, reverse=True)[0]
    sample_df = spark.read.csv(os.path.join(STREAM_PATH, latest_file), header=True, inferSchema=True)
    return sample_df.schema

# 1. Define Stream
stream_df = spark.readStream \
    .schema(get_streaming_schema()) \
    .option("header", True) \
    .option("maxFilesPerTrigger", 1) \
    .csv(f"{STREAM_PATH}*.csv")

# 2. Transformations
processed_stream = stream_df \
    .withColumn("Issued", to_date(col("Issued"))) \
    .withColumn("year", year(col("Issued"))) \
    .withColumn("FSA", upper(substring(col("Licence Address Line 3"), 1, 3))) \
    .filter((col("year") >= 2000) & (col("year") <= 2025)) \
    .select("FSA", "Category", "year")

# 3. Start Stream
query = processed_stream.writeStream \
    .format("parquet") \
    .outputMode("append") \
    .option("checkpointLocation", CHECKPOINT_PATH) \
    .option("path", OUTPUT_PATH) \
    .trigger(processingTime="10 seconds") \
    .start()

print("🌊 Stream processing active...")

🌊 Stream processing active...


In [ ]:
stream_df = stream_df.withColumn("Issued", to_date(col("Issued")))
stream_df = stream_df.withColumn("year", year(col("Issued")))

# Create FSA
stream_df = stream_df.withColumn(
    "FSA",
    upper(substring(col("Licence Address Line 3"), 1, 3))
)

# Filter valid years
stream_df = stream_df.filter((col("year") >= 2000) & (col("year") <= 2025))

In [ ]:
stream_clean = stream_df.select("FSA", "Category", "year")

In [ ]:
output_path = "/content/drive/MyDrive/Big_Data_Group/Group_Project/output/"
checkpoint_path = "/content/drive/MyDrive/Big_Data_Group/Group_Project/checkpoint_stream/"

query = stream_clean.writeStream \
    .format("parquet") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_path) \
    .option("path", output_path) \
    .trigger(processingTime="10 seconds") \
    .start()

In [ ]:
print(query.status)

{'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}


In [ ]:
# query.stop()

In [ ]:
final_df = spark.read.parquet(
    "/content/drive/MyDrive/Big_Data_Group/Group_Project/output/"
)

final_df.show(5)

+---+--------------------+----+
|FSA|            Category|year|
+---+--------------------+----+
|M9N|PRIVATE TRANSPORT...|2018|
|M3J|PRIVATE TRANSPORT...|2017|
|L5A|PRIVATE TRANSPORT...|2018|
|M8Z|PRIVATE TRANSPORT...|2017|
|M2N|PRIVATE TRANSPORT...|2020|
+---+--------------------+----+
only showing top 5 rows


In [ ]:
final_df = final_df.groupBy("FSA", "Category", "year") \
    .count() \
    .withColumnRenamed("count", "business_count")

In [ ]:
final_df = final_df.withColumn(
    "business_density_score",
    col("business_count") / 100
)

In [ ]:
final_df.FSA

In [ ]:
final_pd = final_df.toPandas()

final_pd.head()

,FSA,Category,year,business_count,business_density_score
0,M8Y,PRIVATE PARKING ENFORCEMENT AGENCY,2015,8,0.08
1,M1R,DRIVING SCHOOL OPERATOR (B),2012,16,0.16
2,M4M,DRIVING SCHOOL OPERATOR (B),2002,8,0.08
3,M1V,DRIVING SCHOOL OPERATOR (B),2016,8,0.08
4,M9L,DRIVING SCHOOL OPERATOR (B),2018,8,0.08


In [ ]:
# Convert FSA to neighborhood id
base_path = "/content/drive/MyDrive/Big_Data_Group/Group_Project/src_data"

In [ ]:
geo_path = f"{base_path}/Neighbourhoods-4326.geojson"

with open(geo_path) as f:
    geo = json.load(f)

geo_rows = []

for feature in geo["features"]:
    props = feature["properties"]
    geom = shape(feature["geometry"])
    centroid = geom.centroid

    geo_rows.append({
        "neighborhood_id": int(props["AREA_SHORT_CODE"]),
        "neighborhood_name": props["AREA_NAME"],
        "latitude": centroid.y,
        "longitude": centroid.x
    })

geo_pd = pd.DataFrame(geo_rows)
geo_spark = spark.createDataFrame(geo_pd)

In [ ]:
# Pull population and income data
excel_path = f"{base_path}/neighbourhood-profiles-2021-158-model.xlsx"

excel_df = pd.read_excel(excel_path)

In [ ]:
# Find the EXACT row (should return 1 row)
pop_row = excel_df[excel_df.iloc[:,0].str.contains("Population", case=False, na=False)]
income_row = excel_df[excel_df.iloc[:,0].str.contains("Median.*income", case=False, na=False)]

print(pop_row.shape)
print(income_row.shape)

(54, 159)
(28, 159)


In [ ]:
excel_df.iloc[:, 0].unique()[:50]

array(['Neighbourhood Number', 'TSNS 2020 Designation',
       'Total - Age groups of the population - 25% sample data',
       '  0 to 14 years', '    0 to 4 years', '    5 to 9 years',
       '    10 to 14 years', '  15 to 64 years', '    15 to 19 years',
       '    20 to 24 years', '    25 to 29 years', '    30 to 34 years',
       '    35 to 39 years', '    40 to 44 years', '    45 to 49 years',
       '    50 to 54 years', '    55 to 59 years', '    60 to 64 years',
       '  65 years and over', '    65 to 69 years', '    70 to 74 years',
       '    75 to 79 years', '    80 to 84 years',
       '    85 years and over', '      85 to 89 years',
       '      90 to 94 years', '      95 to 99 years',
       '      100 years and over',
       'Total - Distribution (%) of the population by broad age groups - 25% sample data',
       'Average age of the population', 'Median age of the population',
       'Total - Persons in private households - 25% sample data',
       '  Total - Perso

In [ ]:
excel_df[excel_df.iloc[:,0].str.contains("Total - Persons in private households", case=False, na=False)]

,Neighbourhood Name,West Humber-Clairville,Mount Olive-Silverstone-Jamestown,Thistletown-Beaumond Heights,Rexdale-Kipling,Elms-Old Rexdale,Kingsview Village-The Westway,Willowridge-Martingrove-Richview,Humber Heights-Westmount,Edenbridge-Humber Valley,...,Harbourfront-CityPlace,St Lawrence-East Bayfront-The Islands,Church-Wellesley,Downtown Yonge East,Bay-Cloverhill,Yonge-Bay Corridor,Junction-Wallace Emerson,Dovercourt Village,North Toronto,South Eglinton-Davisville
35,Total - Persons in private households - 25% sa...,33300,31345,9850,10375,9355,22005,22445,10005,15190,...,28135,31285,22320,17700,16670,12645,23180,12380,15885,22735


In [ ]:
# excel_df.iloc[:,0][excel_df.iloc[:,0].str.contains("income", case=False, na=False)]

income_row = excel_df[
    excel_df.iloc[:,0].str.contains(
        "Median total income in 2020  among recipients", case=False, na=False
    )
]
income_row

,Neighbourhood Name,West Humber-Clairville,Mount Olive-Silverstone-Jamestown,Thistletown-Beaumond Heights,Rexdale-Kipling,Elms-Old Rexdale,Kingsview Village-The Westway,Willowridge-Martingrove-Richview,Humber Heights-Westmount,Edenbridge-Humber Valley,...,Harbourfront-CityPlace,St Lawrence-East Bayfront-The Islands,Church-Wellesley,Downtown Yonge East,Bay-Cloverhill,Yonge-Bay Corridor,Junction-Wallace Emerson,Dovercourt Village,North Toronto,South Eglinton-Davisville
62,Median total income in 2020 among recipie...,33600,29600,32800,33600,34400,34400,40800,40400,48400,...,58800,58000,42800,45200,39600,44000,41200,38000,46000,52400


In [ ]:
# Extract values (skip first column which is label)
pop_series = pop_row.iloc[0, 1:]
income_series = income_row.iloc[0, 1:]

# Create DataFrame using COLUMN INDEX (this is the ID!)
pop_df = pd.DataFrame({
    "neighborhood_id": range(1, len(pop_series) + 1),
    "population": pop_series.values
})

income_df = pd.DataFrame({
    "neighborhood_id": range(1, len(income_series) + 1),
    "median_income": income_series.values
})

# Merge
demo_df = pop_df.merge(income_df, on="neighborhood_id")
demo_df.head()

,neighborhood_id,population,median_income
0,1,33300,33600
1,2,31345,29600
2,3,9850,32800
3,4,10375,33600
4,5,9355,34400


In [ ]:
# final_df = final_df.join(geo_spark, on="neighborhood_id", how="left")

In [ ]:
geojson_url = "https://raw.githubusercontent.com/farah-hoque/TorontoCovidPopups/refs/heads/main/Toronto_FSA.json"
geojoson_data = requests.get(geojson_url).json()

In [ ]:
fsa_centroids = []

for feature in geojoson_data["features"]:
    fsa_code = feature["properties"]["CFSAUID"]
    polygon = shape(feature["geometry"])
    centroid = polygon.centroid

    fsa_centroids.append({
        "FSA": fsa_code,
        "lon": centroid.x,
        "lat": centroid.y
    })

fsa_centroids_df = pd.DataFrame(fsa_centroids)

In [ ]:
fsa_centroids_df.head()

,FSA,lon,lat
0,L4X,-79.579839,43.617562
1,M1P,-79.270001,43.762143
2,M1R,-79.297171,43.749456
3,M6C,-79.431316,43.691611
4,M6E,-79.450657,43.688633


In [ ]:
def map_to_neighborhood(lat, lon, geo_features):
    point = Point(lon, lat)
    for feature in geo_features:
        polygon = shape(feature["geometry"])
        if polygon.contains(point):
            return int(feature["properties"]["AREA_SHORT_CODE"])
    return None

mapping = []

for _, row in fsa_centroids_df.iterrows():
    nid = map_to_neighborhood(row["lat"], row["lon"], geo["features"])
    mapping.append({
        "FSA": row["FSA"],
        "neighborhood_id": nid
    })

fsa_to_neighborhood_df = pd.DataFrame(mapping)
fsa_to_neighborhood_df.head()

,FSA,neighborhood_id
0,L4X,NaN
1,M1P,126.0
2,M1R,119.0
3,M6C,106.0
4,M6E,109.0


In [ ]:
# Convert mapping to Spark
fsa_map_spark = spark.createDataFrame(fsa_to_neighborhood_df)

# Join FSA → neighborhood
df_with_neighborhood = final_df.join(
    fsa_map_spark,
    on="FSA",
    how="left"
)

# Filter Toronto FSAs
from pyspark.sql.functions import col, sum as spark_sum

df_with_neighborhood = df_with_neighborhood.filter(
    col("FSA").startswith("M")
)

# Remove unmapped
df_with_neighborhood = df_with_neighborhood.filter(
    col("neighborhood_id").isNotNull()
)
df_with_neighborhood.show(5)

+---+--------------------+----+--------------+----------------------+---------------+
|FSA|            Category|year|business_count|business_density_score|neighborhood_id|
+---+--------------------+----+--------------+----------------------+---------------+
|M1P|DRIVING INSTRUCTO...|2016|             8|                  0.08|          126.0|
|M1P|MASTER HEATING IN...|2013|             8|                  0.08|          126.0|
|M1P|    DRAIN CONTRACTOR|2014|             8|                  0.08|          126.0|
|M1P|TEMPORARY SIGN - ...|2014|            56|                  0.56|          126.0|
|M1P|SECOND HAND SALVA...|2008|             8|                  0.08|          126.0|
+---+--------------------+----+--------------+----------------------+---------------+
only showing top 5 rows


In [ ]:
agg_df = df_with_neighborhood.groupBy("neighborhood_id").agg(
    spark_sum("business_count").alias("total_active_businesses")
)
agg_df.show(5)

+---------------+-----------------------+
|neighborhood_id|total_active_businesses|
+---------------+-----------------------+
|           67.0|                  14056|
|           70.0|                  13936|
|          112.0|                  12416|
|           88.0|                  12768|
|           98.0|                  13272|
+---------------+-----------------------+
only showing top 5 rows


In [ ]:
demo_spark = spark.createDataFrame(demo_df)

In [ ]:
final_df = agg_df.join(geo_spark, on="neighborhood_id", how="left")
final_df = final_df.join(demo_spark, on="neighborhood_id", how="left")

In [ ]:
final_pd = final_df.toPandas()
final_pd.head()

,neighborhood_id,total_active_businesses,neighborhood_name,latitude,longitude,population,median_income
0,67.0,14056,Playter Estates-Danforth,43.679700,-79.354887,12750.0,33200.0
1,70.0,13936,South Riverdale,43.649203,-79.335617,18120.0,32400.0
2,112.0,12416,Beechborough-Greenbrook,43.693216,-79.479472,17110.0,32000.0
3,88.0,12768,High Park North,43.657565,-79.466302,20080.0,67500.0
4,98.0,13272,Rosedale-Moore Park,43.682806,-79.379650,10015.0,37200.0


In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Group by neighborhood_id and Category to count businesses per category in each neighborhood
business_category_counts = df_with_neighborhood.groupBy("neighborhood_id", "Category").agg(
    count("*").alias("category_business_count")
)

# Define a window specification partitioned by neighborhood_id, ordered by category_business_count in descending order
window_spec = Window.partitionBy("neighborhood_id").orderBy(col("category_business_count").desc())

# Apply the window function to rank categories within each neighborhood
ranked_categories = business_category_counts.withColumn("rank", row_number().over(window_spec))

# Filter for the top category (rank 1) for each neighborhood
most_common_category_per_neighborhood = ranked_categories.filter(col("rank") == 1).drop("rank")

# Optionally, join with geo_spark to add neighborhood names
most_common_category_with_names = most_common_category_per_neighborhood.join(
    geo_spark.select("neighborhood_id", "neighborhood_name"),
    on="neighborhood_id",
    how="left"
)

# Show the result
print("Most common business category per neighborhood:")
display(most_common_category_with_names.toPandas().head(10))

Most common business category per neighborhood:


,neighborhood_id,Category,category_business_count,neighborhood_name
0,1.0,EATING OR DRINKING ESTABLISHMENT,26,West Humber-Clairville
1,2.0,TAXICAB OWNER,26,Mount Olive-Silverstone-Jamestown
2,7.0,TAKE-OUT OR RETAIL FOOD ESTABLISHMENT,51,Willowridge-Martingrove-Richview
3,9.0,TAKE-OUT OR RETAIL FOOD ESTABLISHMENT,26,Edenbridge-Humber Valley
4,12.0,TAKE-OUT OR RETAIL FOOD ESTABLISHMENT,26,Markland Wood
5,15.0,TAKE-OUT OR RETAIL FOOD ESTABLISHMENT,26,Kingsway South
6,16.0,TAKE-OUT OR RETAIL FOOD ESTABLISHMENT,26,Stonegate-Queensway
7,18.0,TAKE-OUT OR RETAIL FOOD ESTABLISHMENT,26,New Toronto
8,20.0,EATING OR DRINKING ESTABLISHMENT,26,Alderwood
9,21.0,EATING OR DRINKING ESTABLISHMENT,26,Humber Summit


In [ ]:
merged_final_df = final_pd.merge(
    most_common_category_with_names.select("neighborhood_id", "Category", "category_business_count").toPandas(),
    on="neighborhood_id",
    how="left"
)

print("Combined DataFrame with most common business category per neighborhood:")
display(merged_final_df.head())

Combined DataFrame with most common business category per neighborhood:


,neighborhood_id,total_active_businesses,neighborhood_name,latitude,longitude,population,median_income,Category,category_business_count
0,67.0,14056,Playter Estates-Danforth,43.679700,-79.354887,12750.0,33200.0,EATING OR DRINKING ESTABLISHMENT,26
1,70.0,13936,South Riverdale,43.649203,-79.335617,18120.0,32400.0,EATING OR DRINKING ESTABLISHMENT,26
2,112.0,12416,Beechborough-Greenbrook,43.693216,-79.479472,17110.0,32000.0,EATING OR DRINKING ESTABLISHMENT,26
3,88.0,12768,High Park North,43.657565,-79.466302,20080.0,67500.0,TAKE-OUT OR RETAIL FOOD ESTABLISHMENT,26
4,98.0,13272,Rosedale-Moore Park,43.682806,-79.379650,10015.0,37200.0,EATING OR DRINKING ESTABLISHMENT,52


In [ ]:
output_csv_path = "/content/drive/MyDrive/Big_Data_Group/Group_Project/final_dashboard.csv"
merged_final_df.to_csv(output_csv_path, index=False)
print(f"DataFrame saved to {output_csv_path}")

DataFrame saved to /content/drive/MyDrive/Big_Data_Group/Group_Project/final_dashboard.csv
